In [5]:
from datasets import Dataset
import json
import pyarrow as pa
import pandas as pd
import numpy as np
from transformers import T5Tokenizer, T5Model, pipeline
import nltk, evaluate
from nltk import sent_tokenize
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [3]:
pipe = pipeline("text2text-generation", model="hawalurahman/idt5-base-qaqg-v1.42-SQuAD-id")

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
/home/halim/.virtualenvs/experiment/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use cuda:0


In [4]:
pipe("Generate question: Jakarta adalah ibukota Indonesia. [SEP] Jakarta")

[{'generated_text': 'Apa ibu kota Indonesia?'}]

In [18]:
def load_data(filepath):
    if filepath.endswith('.jsonl'):
        data = []
        with open(filepath) as f:
            for line in f:
                data.append(json.loads(line))
            data = [item for item in data if item['is_impossible'] == False]
        print("data loaded")
        return data
    else:
        with open(filepath) as f:
            data = json.load(f)
        print("data loaded")
    return data

def answer_extraction(context):
    inputs = f"extract answer: {context}"
    outputs = pipe(inputs)
    answer = outputs[0]['generated_text']
    return answer

def question_generation(context, answer):
    inputs = f"Generate question: {context} [SEP] {answer}"
    outputs = pipe(inputs)
    question = outputs[0]['generated_text']
    return question

In [19]:
squad_test = load_data('../data/squad-dev.jsonl')
squad_test

data loaded


[{'context': 'Bangsa Normandia (Norman: Nourmands; Prancis: Normandia; Latin: Normanni) adalah orang-orang yang pada abad ke-10 dan ke-11 memberikan nama mereka kepada Normandia, sebuah wilayah di Perancis. Mereka diturunkan dari Norse ("Norman" berasal dari "Norseman") perampok dan bajak laut dari Denmark, Islandia dan Norwegia yang, di bawah pemimpin mereka Rollo, setuju untuk bersumpah setia kepada Raja Charles III dari Francia Barat. Melalui generasi-generasi asimilasi dan percampuran dengan penduduk asli Frank dan Romawi-Galia, keturunan mereka akan secara bertahap bergabung dengan budaya yang berbasis di Carolingian di Francia Barat. Identitas budaya dan etnis yang berbeda dari Normandia awalnya muncul pada paruh pertama abad ke-10, dan itu terus berkembang selama berabad-abad berikutnya.',
  'question': 'Di negara apa Normandia berada?',
  'answer_alternatives': ['Perancis', 'Perancis', 'Perancis', 'Perancis'],
  'answer': ['Perancis'],
  'is_impossible': False,
  'id': '56ddde6

In [20]:
tydiqa_test = load_data('../data/tydiqa-preprocesed-eval.json')
tydiqa_test

data loaded


[{'context': "Kolumbus bukanlah orang pertama yang tiba di Amerika, yang ia dapati sudah diduduki. Ia juga bukan orang Eropa pertama yang sampai ke benua itu karena sekarang telah diakui secara meluas bahwa orang-orang Viking dari Eropa Utara telah berkunjung ke Amerika Utara pada abad ke 11 dan mendirikan koloni L'Anse aux Meadows untuk jangka waktu singkat. Terdapat perkiraan bahwa pelayar yang tidak dikenali pernah melawat ke Amerika sebelum Kolumbus dan membekalkannya dengan sumber untuk kejayaannya. Terdapat juga banyak teori mengenai ekspedisi ke Amerika oleh berbagai orang sepanjang masa itu.",
  'question': 'Siapakah yang menemuka benua Amerika ?',
  'answer': 'orang-orang Viking dari Eropa Utara',
  'answer_start': 193},
 {'context': 'Kabupaten Donggala (English: Donggala Regency), adalah sebuah kabupaten di provinsi Sulawesi Tengah, Indonesia. Ibu kota kabupaten sekaligus pusat administrasi terletak di Kota Donggala. Kabupaten ini mempunyai luas sebesar 4275,08km² dan berpend

In [34]:
context_only = pd.DataFrame([item['context'] for item in squad_test])
unique_context = list(context_only[0].unique())
unique_context

['Bangsa Normandia (Norman: Nourmands; Prancis: Normandia; Latin: Normanni) adalah orang-orang yang pada abad ke-10 dan ke-11 memberikan nama mereka kepada Normandia, sebuah wilayah di Perancis. Mereka diturunkan dari Norse ("Norman" berasal dari "Norseman") perampok dan bajak laut dari Denmark, Islandia dan Norwegia yang, di bawah pemimpin mereka Rollo, setuju untuk bersumpah setia kepada Raja Charles III dari Francia Barat. Melalui generasi-generasi asimilasi dan percampuran dengan penduduk asli Frank dan Romawi-Galia, keturunan mereka akan secara bertahap bergabung dengan budaya yang berbasis di Carolingian di Francia Barat. Identitas budaya dan etnis yang berbeda dari Normandia awalnya muncul pada paruh pertama abad ke-10, dan itu terus berkembang selama berabad-abad berikutnya.',
 'Dinasti Norman memiliki dampak politik, budaya, dan militer yang besar pada Eropa abad pertengahan dan bahkan Timur Dekat. Bangsa Normandia terkenal karena semangat bela diri mereka dan akhirnya karena 

In [38]:
unique_context

['Bangsa Normandia (Norman: Nourmands; Prancis: Normandia; Latin: Normanni) adalah orang-orang yang pada abad ke-10 dan ke-11 memberikan nama mereka kepada Normandia, sebuah wilayah di Perancis. Mereka diturunkan dari Norse ("Norman" berasal dari "Norseman") perampok dan bajak laut dari Denmark, Islandia dan Norwegia yang, di bawah pemimpin mereka Rollo, setuju untuk bersumpah setia kepada Raja Charles III dari Francia Barat. Melalui generasi-generasi asimilasi dan percampuran dengan penduduk asli Frank dan Romawi-Galia, keturunan mereka akan secara bertahap bergabung dengan budaya yang berbasis di Carolingian di Francia Barat. Identitas budaya dan etnis yang berbeda dari Normandia awalnya muncul pada paruh pertama abad ke-10, dan itu terus berkembang selama berabad-abad berikutnya.',
 'Dinasti Norman memiliki dampak politik, budaya, dan militer yang besar pada Eropa abad pertengahan dan bahkan Timur Dekat. Bangsa Normandia terkenal karena semangat bela diri mereka dan akhirnya karena 

In [42]:
from tqdm import tqdm 

generated_answers = [answer_extraction(item) for item in tqdm(unique_context[:10])]

100%|██████████| 10/10 [00:04<00:00,  2.14it/s]


In [43]:
generated_answers

['Normandia [SEP] Normandia [SEP] Normandia [SEP] Rollo [SEP] Carolingian',
 'Kristen [SEP] Gallo-Romantis [SEP] Richard I [SEP] 1066 [SEP] Bohemond I',
 'Normans / Normanz [SEP] Normant, normand Perancis modern, [SEP] Old Low Franconian Nortmann "Northman" atau langsung dari Old Norse Norðma ,r [SEP] Nortmannus, Normannus, atau Nordmannus',
 'wanita lokal dan properti pribadi [SEP] 911 [SEP] Raja Charles III dari Francia Barat dan penguasa terkenal Viking Rollo [SEP] bagian utara Normandia Atas sekarang yang sekarang sampai ke sungai Seine, tetapi Kadipaten pada akhirnya akan membentang ke barat melewati Seine. Wilayah itu kira-kira setara dengan provinsi tua Rouen, dan mereproduksi struktur administrasi Romawi Gallia Lugdunensis II (bagian dari bekas Gallia Lugdunensis).',
 'Picardy atau <0xC3>le-de-France, yang dianggap "Frankish". [SEP] dibagi antara koloni di timur (Roumois dan Pays de Caux) di sekitar lembah Seine rendah dan di barat di Semenanjung Cotentin, [SEP] hampir tidak ad

In [46]:
generated_question = [question_generation(item['context'], item['answer'][0]) for item in tqdm(squad_test[:10])]

100%|██████████| 10/10 [00:01<00:00,  9.37it/s]


In [ ]:
generated_question

['Di negara manakah Normandia berada?',
 'Pada abad berapa Normandia memberikan nama mereka kepada Normandia?',
 'Dari mana asal Normandia?',
 'Siapa pemimpin Normandia?',
 'Kapan Normandia pertama kali memberikan nama mereka kepada Normandia?',
 'Siapakah adipati Normandia?',
 'Siapa yang memimpin Kadipaten Normandia?',
 'Bangsa Normandia terkenal karena semangat bela diri mereka dan akhirnya karena kesalehan apa?',
 'Nortmannus, Normannus, atau Nordmannus berarti Nordmannus, Normannus, atau Nordmannus dalam bahasa apa?',
 'Kapan Nortmannus, Normannus, atau Nordmannus ditemukan dalam Abad Pertengahan Latin?']